In [1]:
!pip install ramantune


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
from ramantune.pipeline.raman_pipeline import RamanPipeline

from ramantune.search.search_space import DenoiserSpace, BaselineSpace, NormalizerSpace, ClassifierSpace, FeatureSelectionSpace
from ramantune.utils.config import DENOISING_STR, BASELINE_STR, NORMALIZE_STR, FEATURE_SELECTION_STR, CLASSIFIER_STR
from ramantune.search.strategies import GridSearchStrategy
from ramantune.search import RamanSearch

from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.datasets import make_regression

df = pd.read_csv("bin/ovarian_small.csv")
groups = df['patient']

y = df['label'].values
X = df.drop(columns=['label', 'patient'])

# Create a fake regression task and get y values
_, y = make_regression(n_samples=len(X), n_features=2, n_informative=2, random_state=0)

In [7]:
def setup_param_grid():
  denoiser_list = [
        DenoiserSpace("savgol", {"window_length": [7], "polyorder": [3]}),
        DenoiserSpace(None) # No Denoising
  ]

  baseline_list = [
      BaselineSpace("asls", {"lam": [100]}),
  ]

  normalization_list = [
      NormalizerSpace("vector"),
  ]

  feature_selection_list = [
      FeatureSelectionSpace(None), # No feature selection
      FeatureSelectionSpace(PCA(),{"n_components": [0.90, 10]}),
  ]

  regressor_list = [
      ClassifierSpace(SVR(),{"C": [0.1, 10], "kernel": ["rbf", "linear"], "gamma": ["scale"]}),
      ClassifierSpace(RandomForestRegressor(),{"n_estimators": [100, 200]})
  ]

  param_list = {
      DENOISING_STR: denoiser_list,
      BASELINE_STR: baseline_list,
      NORMALIZE_STR: normalization_list,
      FEATURE_SELECTION_STR: feature_selection_list,
      CLASSIFIER_STR: regressor_list
  }

  return param_list

In [11]:
estimator = RamanPipeline()
param_grid = setup_param_grid()

search = RamanSearch(estimator=estimator,
                     research_strategy=GridSearchStrategy(),
                     param_grid=param_grid,
                     cv=GroupKFold(n_splits=2, random_state=42, shuffle=True),
                     return_train_score=False,
                     n_jobs=5,
                     verbose=10,
                     scoring={"r2":"r2","mean_squared_error":"neg_mean_squared_error"},
                     refit="r2")

res = search.fit(X, y, groups=groups)


Fitting 2 folds for each of 36 candidates, totalling 72 fits
[TIME] fit took 7.44 seconds


In [12]:
print(search.get_best_params())
print(search.get_best_score())

{'baseline__algorithm': 'asls', 'baseline__lam': 100, 'classifier__algorithm': RandomForestRegressor(), 'classifier__n_estimators': 100, 'denoising__algorithm': 'savgol', 'denoising__polyorder': 3, 'denoising__window_length': 7, 'feature__algorithm': PCA(), 'feature__n_components': 0.9, 'normalize__algorithm': 'vector'}
0.06107016101624252


In [13]:
result_cv = search.get_cv_results(file_path=f"result_add_preprocessing.csv",
                          return_split_scores=False,
                          return_combined_params=True,
                          round_values=True)

In [14]:
result_cv

,denoising,baseline,normalize,feature,classifier,mean_fit_time,std_fit_time,mean_score_time,std_score_time,split0_test_r2,...,split0_test_mean_squared_error,split1_test_mean_squared_error,mean_test_mean_squared_error,std_test_mean_squared_error,rank_test_mean_squared_error,split0_test_patient_accuracy,split1_test_patient_accuracy,mean_test_patient_accuracy,std_test_patient_accuracy,rank_test_patient_accuracy
0,"savgol(polyorder=3.0,window_length=7.0)",asls(lam=100),vector,None,"SVR(C=0.1,kernel=rbf)",0.1513,0.0113,0.1020,0.0053,-0.0376,...,-6260.4060,-6772.8206,-6516.6133,256.2073,15,NaN,NaN,NaN,NaN,1
1,"savgol(polyorder=3.0,window_length=7.0)",asls(lam=100),vector,None,"SVR(C=0.1,kernel=linear)",0.1520,0.0021,0.1019,0.0046,-0.0375,...,-6260.3097,-6772.6518,-6516.4808,256.1711,9,NaN,NaN,NaN,NaN,1
2,"savgol(polyorder=3.0,window_length=7.0)",asls(lam=100),vector,None,"SVR(C=10.0,kernel=rbf)",0.1306,0.0358,0.0970,0.0020,-0.0377,...,-6261.5987,-6790.7576,-6526.1781,264.5794,25,NaN,NaN,NaN,NaN,1
3,"savgol(polyorder=3.0,window_length=7.0)",asls(lam=100),vector,None,"SVR(C=10.0,kernel=linear)",0.0925,0.0103,0.1087,0.0075,-0.0385,...,-6266.0075,-6780.6003,-6523.3039,257.2964,21,NaN,NaN,NaN,NaN,1
4,"savgol(polyorder=3.0,window_length=7.0)",asls(lam=100),vector,None,RandomForestRegressor(n_estimators=100.0),0.3951,0.0221,0.0980,0.0025,-0.1361,...,-6854.8762,-6595.3144,-6725.0953,129.7809,31,NaN,NaN,NaN,NaN,1
5,"savgol(polyorder=3.0,window_length=7.0)",asls(lam=100),vector,None,RandomForestRegressor(n_estimators=200.0),0.6739,0.0086,0.1229,0.0023,-0.0339,...,-6238.5438,-6350.1050,-6294.3244,55.7806,3,NaN,NaN,NaN,NaN,1
6,"savgol(polyorder=3.0,window_length=7.0)",asls(lam=100),vector,PCA(n_components=0.9),"SVR(C=0.1,kernel=rbf)",0.1034,0.0282,0.0918,0.0099,-0.0378,...,-6261.9102,-6772.3753,-6517.1427,255.2325,17,NaN,NaN,NaN,NaN,1
7,"savgol(polyorder=3.0,window_length=7.0)",asls(lam=100),vector,PCA(n_components=10.0),"SVR(C=0.1,kernel=rbf)",0.0661,0.0010,0.0868,0.0060,-0.0379,...,-6262.4263,-6772.7015,-6517.5639,255.1376,20,NaN,NaN,NaN,NaN,1
8,"savgol(polyorder=3.0,window_length=7.0)",asls(lam=100),vector,PCA(n_components=0.9),"SVR(C=0.1,kernel=linear)",0.0951,0.0289,0.0932,0.0015,-0.0375,...,-6260.3183,-6772.6490,-6516.4837,256.1653,10,NaN,NaN,NaN,NaN,1
9,"savgol(polyorder=3.0,window_length=7.0)",asls(lam=100),vector,PCA(n_components=10.0),"SVR(C=0.1,kernel=linear)",0.1309,0.0024,0.0921,0.0014,-0.0375,...,-6260.3120,-6772.6909,-6516.5014,256.1895,11,NaN,NaN,NaN,NaN,1
